# 05 — Evaluación Final
**Fase 5 · CRISP-DM** | Proyecto: Predicción de Stroke

UNAM · Facultad de Ingeniería · Minería de Datos · Semestre 2027-1

---

Evalúa los modelos entrenados sobre el conjunto de **prueba** (sin SMOTE).

Genera:
- Tabla comparativa: AUC-ROC, Recall, F1-score, Accuracy
- Curvas ROC de ambos modelos
- Matrices de confusión
- Importancia de características (Random Forest)
- Verificación de umbrales del anteproyecto

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, roc_curve,
)

%matplotlib inline

DATA_DIR = os.path.join(os.getcwd(), "data")
OUT_DIR  = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Cargar conjunto de prueba
test_df = pd.read_csv(os.path.join(DATA_DIR, "processed_test.csv"))
X_test  = test_df.drop(columns=["stroke"])
y_test  = test_df["stroke"]
print(f"Test cargado: {X_test.shape[0]} muestras")

# Cargar modelos
MODEL_FILES = {
    "Logistic Regression": "model_logistic_regression.pkl",
    "Random Forest":       "model_random_forest.pkl",
}
models = {}
for name, fname in MODEL_FILES.items():
    with open(os.path.join(DATA_DIR, fname), "rb") as f:
        models[name] = pickle.load(f)
print("Modelos cargados:", list(models.keys()))

## 5.1 Tabla de métricas

In [ ]:
rows = []
for name, model in models.items():
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        "Modelo":    name,
        "AUC-ROC":  round(roc_auc_score(y_test, y_proba), 4),
        "Recall":   round(recall_score(y_test, y_pred),   4),
        "F1-score": round(f1_score(y_test, y_pred),       4),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    })

metrics_df = pd.DataFrame(rows)
metrics_df

## 5.2 Verificación de umbrales del anteproyecto

In [ ]:
print(f"{'Modelo':<25} | AUC>0.85 | Recall>0.75 | F1>0.60")
print("-" * 60)
for _, row in metrics_df.iterrows():
    auc_ok    = "✓" if row["AUC-ROC"]  > 0.85 else "✗"
    recall_ok = "✓" if row["Recall"]   > 0.75 else "✗"
    f1_ok     = "✓" if row["F1-score"] > 0.60 else "✗"
    print(f"{row['Modelo']:<25} |    {auc_ok}({row['AUC-ROC']}) |     {recall_ok}({row['Recall']}) |  {f1_ok}({row['F1-score']})")

## 5.3 Curvas ROC

In [ ]:
colors = {"Logistic Regression": "#3b82d4", "Random Forest": "#7c5cd8"}
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=colors.get(name, "gray"))
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random (AUC=0.50)")
ax.set_xlabel("Tasa de falsos positivos (FPR)")
ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
ax.set_title("Curvas ROC — Conjunto de prueba")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "06_roc_curves.png"), dpi=150)
plt.show()

## 5.4 Matrices de confusión

In [ ]:
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
if n == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred: No stroke", "Pred: Stroke"],
                yticklabels=["Real: No stroke", "Real: Stroke"])
    ax.set_title(f"Matriz de confusión\n{name}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "07_confusion_matrices.png"), dpi=150)
plt.show()

## 5.5 Importancia de características (Random Forest)

In [ ]:
rf_model = models["Random Forest"]
feat_df = pd.DataFrame({
    "feature":    X_test.columns,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_df["feature"][::-1], feat_df["importance"][::-1], color="#3b82d4")
ax.set_xlabel("Importancia (Gini)")
ax.set_title("Top 15 variables más importantes\n(Random Forest)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "08_feature_importance.png"), dpi=150)
plt.show()

---
✅ **Fase de evaluación completada.** Figuras guardadas en `outputs/`.